# ЛР 4, вариант 2: обратная матрица. Замеры для отчёта

Ноутбук делает всё для раздела «Результаты» отчёта:
1. записывает в Colab программы: сдаваемую `lab4.cu`, профилирующую `lab4_prof.cu` (замеры через `cudaEvent`), CPU-версию `lab4_cpu.cpp` и генератор матриц `gen_matrix.py`;
2. проверяет корректность GPU и CPU по `numpy.linalg.inv`;
3. строит таблицу «CPU и GPU, время и ускорение» для разных n;
4. показывает, на что тратится время по фазам алгоритма;
5. исследует влияние конфигурации сетки и блока ядра `update`;
6. рисует график и запускает профайлер `nsys` (замена `nvprof`).

**Как запускать:** *Среда выполнения → Сменить тип среды выполнения → T4 GPU*, затем *Среда выполнения → Выполнить все*.
Размеры и число повторов задаются в ячейке «Параметры».

In [ ]:
%%shell
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
nvcc --version | tail -2
g++ --version | head -1
lscpu | grep -E "Model name|^CPU\(s\)"

## 1. Файлы программ

In [ ]:
%%writefile lab4.cu
// ЛР №4, вариант 2. Вычисление обратной матрицы методом Гаусса-Жордана
// с выбором главного элемента по столбцу (CUDA + Thrust только для поиска максимума).
// Сборка: nvcc -Werror cross-execution-space-call -lm -o lab4 lab4.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <thrust/extrema.h>
#include <thrust/device_ptr.h>

#define CSC(call)                                                          \
do {                                                                       \
    cudaError_t res = (call);                                              \
    if (res != cudaSuccess) {                                              \
        fprintf(stderr, "ERROR: CUDA failure in %s:%d. Message: %s\n",     \
                __FILE__, __LINE__, cudaGetErrorString(res));              \
        exit(0);                                                           \
    }                                                                      \
} while (0)

#define MAX_N 10000     // n <= 10^4
#define CHUNK 64        // число строк, обрабатываемых за раз при вводе/выводе

// Конфигурации запуска фиксированы и не зависят от размера матрицы.
const int  GRID1 = 256, BLOCK1 = 256;                // одномерные ядра
const dim3 GRID2(16, 16), BLOCK2(32, 32);            // двумерное ядро

// Компаратор для Thrust: сравнение по модулю.
struct abs_less {
    __host__ __device__ bool operator()(double a, double b) const {
        return fabs(a) < fabs(b);
    }
};

// ---------------------------------------------------------------------------
// Матрица хранится по столбцам: элемент (i, j) -- A[j * n + i].
// Столбец лежит в памяти подряд, поэтому поиск главного элемента по столбцу
// (Thrust) идёт по непрерывному участку, а потоки с соседними номерами по x
// обращаются к соседним адресам (объединение запросов к глобальной памяти).
// ---------------------------------------------------------------------------

// Обмен строк r1 и r2 во всех столбцах.
__global__ void swap_rows(double *A, int n, int r1, int r2) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    for (int j = idx; j < n; j += offset) {
        double t = A[(long long)j * n + r1];
        A[(long long)j * n + r1] = A[(long long)j * n + r2];
        A[(long long)j * n + r2] = t;
    }
}

// Обмен столбцов c1 и c2 (столбцы непрерывны в памяти).
__global__ void swap_cols(double *A, int n, int c1, int c2) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += offset) {
        double t = A[(long long)c1 * n + i];
        A[(long long)c1 * n + i] = A[(long long)c2 * n + i];
        A[(long long)c2 * n + i] = t;
    }
}

// Копия ведущего столбца k и ведущей строки k (делённой на ведущий элемент).
// Копии нужны, чтобы основное ядро не читало то, что само же перезаписывает.
__global__ void prepare(const double *A, double *col, double *row, int n, int k, double p) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    for (int t = idx; t < n; t += offset) {
        col[t] = A[(long long)k * n + t];        // A[t][k]
        row[t] = A[(long long)t * n + k] / p;    // A[k][t] / p
    }
}

// Один шаг Гаусса-Жордана "на месте" (обратная матрица замещает исходную):
//   A[k][k]  = 1 / p
//   A[k][j]  = A[k][j] / p                      (j != k)
//   A[i][k]  = -A[i][k] / p                     (i != k)
//   A[i][j] -= A[i][k] * A[k][j] / p            (i != k, j != k)
// Двумерная сетка: x -- строки (соседние адреса), y -- столбцы.
__global__ void update(double *A, const double *col, const double *row, int n, int k, double p) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int idy = blockIdx.y * blockDim.y + threadIdx.y;
    int offsetx = blockDim.x * gridDim.x;
    int offsety = blockDim.y * gridDim.y;
    for (int j = idy; j < n; j += offsety) {
        double rj = row[j];
        for (int i = idx; i < n; i += offsetx) {
            long long pos = (long long)j * n + i;
            if (i == k && j == k)  A[pos] = 1.0 / p;
            else if (i == k)       A[pos] = rj;
            else if (j == k)       A[pos] = -col[i] / p;
            else                   A[pos] -= col[i] * rj;
        }
    }
}

// ---------------------------------------------------------------------------
// Быстрый ввод: чтение блоками и разбор токенов (scanf слишком медленный
// для 10^8 чисел).
// ---------------------------------------------------------------------------
static unsigned char ibuf[1 << 16];
static size_t ipos = 0, ilen = 0;

static int next_char() {
    if (ipos == ilen) {
        ilen = fread(ibuf, 1, sizeof(ibuf), stdin);
        ipos = 0;
        if (ilen == 0) return -1;
    }
    return ibuf[ipos++];
}

static int read_token(char *tok, int cap) {
    int c;
    do { c = next_char(); } while (c == ' ' || c == '\n' || c == '\r' || c == '\t');
    if (c < 0) return 0;
    int len = 0;
    while (c >= 0 && c != ' ' && c != '\n' && c != '\r' && c != '\t') {
        if (len < cap - 1) tok[len++] = (char)c;
        c = next_char();
    }
    tok[len] = '\0';
    return 1;
}

static int read_double(double *x) {
    char tok[128];
    if (!read_token(tok, sizeof(tok))) return 0;
    char *end;
    *x = strtod(tok, &end);
    return *end == '\0';
}

static int read_int(long long *x) {
    char tok[128];
    if (!read_token(tok, sizeof(tok))) return 0;
    char *end;
    *x = strtoll(tok, &end, 10);
    return *end == '\0';
}

int main() {
    long long nll;
    if (!read_int(&nll) || nll < 1 || nll > MAX_N) {
        fprintf(stderr, "ERROR: invalid matrix size (must be 1..%d)\n", MAX_N);
        return 0;
    }
    int n = (int)nll;
    size_t total = (size_t)n * (size_t)n;

    double *h = (double *)malloc(sizeof(double) * total);        // по столбцам
    double *buf = (double *)malloc(sizeof(double) * (size_t)CHUNK * n);
    if (h == NULL || buf == NULL) {
        fprintf(stderr, "ERROR: not enough host memory\n");
        free(h); free(buf);
        return 0;
    }

    // Чтение по строкам блоками по CHUNK строк, запись в хранение по столбцам.
    for (int i0 = 0; i0 < n; i0 += CHUNK) {
        int rows = (n - i0 < CHUNK) ? (n - i0) : CHUNK;
        for (int r = 0; r < rows; r++)
            for (int j = 0; j < n; j++)
                if (!read_double(&buf[(size_t)r * n + j])) {
                    fprintf(stderr, "ERROR: cannot read matrix element (%d, %d)\n", i0 + r, j);
                    free(h); free(buf);
                    return 0;
                }
        for (int j = 0; j < n; j++)
            for (int r = 0; r < rows; r++)
                h[(size_t)j * n + i0 + r] = buf[(size_t)r * n + j];
    }

    double *dev_A = NULL, *dev_col = NULL, *dev_row = NULL;
    CSC(cudaMalloc(&dev_A, sizeof(double) * total));
    CSC(cudaMalloc(&dev_col, sizeof(double) * n));
    CSC(cudaMalloc(&dev_row, sizeof(double) * n));
    CSC(cudaMemcpy(dev_A, h, sizeof(double) * total, cudaMemcpyHostToDevice));

    int *piv = (int *)malloc(sizeof(int) * n);
    if (piv == NULL) {
        fprintf(stderr, "ERROR: not enough host memory\n");
        return 0;
    }

    for (int k = 0; k < n; k++) {
        // Главный элемент по столбцу k среди строк k..n-1 (Thrust).
        thrust::device_ptr<double> first = thrust::device_pointer_cast(dev_A + (size_t)k * n + k);
        thrust::device_ptr<double> last = thrust::device_pointer_cast(dev_A + (size_t)k * n + n);
        thrust::device_ptr<double> it = thrust::max_element(first, last, abs_less());
        int r = k + (int)(it - first);
        piv[k] = r;

        if (r != k) {
            swap_rows<<<GRID1, BLOCK1>>>(dev_A, n, k, r);
            CSC(cudaGetLastError());
        }

        double p;
        CSC(cudaMemcpy(&p, dev_A + (size_t)k * n + k, sizeof(double), cudaMemcpyDeviceToHost));
        if (p == 0.0) {
            fprintf(stderr, "ERROR: matrix is singular\n");
            CSC(cudaFree(dev_A)); CSC(cudaFree(dev_col)); CSC(cudaFree(dev_row));
            free(h); free(buf); free(piv);
            return 0;
        }

        prepare<<<GRID1, BLOCK1>>>(dev_A, dev_col, dev_row, n, k, p);
        CSC(cudaGetLastError());
        update<<<GRID2, BLOCK2>>>(dev_A, dev_col, dev_row, n, k, p);
        CSC(cudaGetLastError());
    }

    // Перестановки строк на шагах прямого хода превращаются в перестановки
    // столбцов результата; применяются в обратном порядке.
    for (int k = n - 1; k >= 0; k--) {
        if (piv[k] != k) {
            swap_cols<<<GRID1, BLOCK1>>>(dev_A, n, k, piv[k]);
            CSC(cudaGetLastError());
        }
    }
    CSC(cudaDeviceSynchronize());

    CSC(cudaMemcpy(h, dev_A, sizeof(double) * total, cudaMemcpyDeviceToHost));
    CSC(cudaFree(dev_A));
    CSC(cudaFree(dev_col));
    CSC(cudaFree(dev_row));

    // Вывод по строкам, блоками по CHUNK строк (чтение по столбцам блоками).
    static char outbuf[1 << 20];
    setvbuf(stdout, outbuf, _IOFBF, sizeof(outbuf));
    for (int i0 = 0; i0 < n; i0 += CHUNK) {
        int rows = (n - i0 < CHUNK) ? (n - i0) : CHUNK;
        for (int j = 0; j < n; j++)
            for (int r = 0; r < rows; r++)
                buf[(size_t)r * n + j] = h[(size_t)j * n + i0 + r];
        for (int r = 0; r < rows; r++)
            for (int j = 0; j < n; j++)
                printf("%.10e%c", buf[(size_t)r * n + j], j + 1 < n ? ' ' : '\n');
    }

    free(h);
    free(buf);
    free(piv);
    return 0;
}


In [ ]:
%%writefile lab4_prof.cu
// ЛР №4, вариант 2. ПРОФИЛИРУЮЩАЯ версия (для отчёта, не для сдачи).
// Алгоритм тот же, что в lab4.cu, плюс замеры времени через cudaEvent и
// настраиваемая конфигурация запуска двумерного ядра update.
//
// Сборка:  nvcc -arch=sm_75 -O2 -o lab4_prof lab4_prof.cu
// Запуск:  ./lab4_prof [--grid GX GY] [--block BX BY] [--noout] [--detail] < matrix.txt
//   --grid / --block  конфигурация ядра update (по умолчанию 16 16 и 32 32)
//   --noout           не печатать обратную матрицу (замер без вывода)
//   --detail          замер по фазам (поиск максимума, обмен строк, подготовка, update);
//                     добавляет накладные расходы на события на каждом шаге
// Результат замеров -- одна строка TIMING=... в stderr.
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <chrono>
#include <thrust/extrema.h>
#include <thrust/device_ptr.h>

#define CSC(call)                                                          \
do {                                                                       \
    cudaError_t res = (call);                                              \
    if (res != cudaSuccess) {                                              \
        fprintf(stderr, "ERROR: CUDA failure in %s:%d. Message: %s\n",     \
                __FILE__, __LINE__, cudaGetErrorString(res));              \
        exit(0);                                                           \
    }                                                                      \
} while (0)

#define MAX_N 10000
#define CHUNK 64

const int GRID1 = 256, BLOCK1 = 256;

struct abs_less {
    __host__ __device__ bool operator()(double a, double b) const {
        return fabs(a) < fabs(b);
    }
};

__global__ void swap_rows(double *A, int n, int r1, int r2) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    for (int j = idx; j < n; j += offset) {
        double t = A[(long long)j * n + r1];
        A[(long long)j * n + r1] = A[(long long)j * n + r2];
        A[(long long)j * n + r2] = t;
    }
}

__global__ void swap_cols(double *A, int n, int c1, int c2) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += offset) {
        double t = A[(long long)c1 * n + i];
        A[(long long)c1 * n + i] = A[(long long)c2 * n + i];
        A[(long long)c2 * n + i] = t;
    }
}

__global__ void prepare(const double *A, double *col, double *row, int n, int k, double p) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int offset = blockDim.x * gridDim.x;
    for (int t = idx; t < n; t += offset) {
        col[t] = A[(long long)k * n + t];
        row[t] = A[(long long)t * n + k] / p;
    }
}

__global__ void update(double *A, const double *col, const double *row, int n, int k, double p) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int idy = blockIdx.y * blockDim.y + threadIdx.y;
    int offsetx = blockDim.x * gridDim.x;
    int offsety = blockDim.y * gridDim.y;
    for (int j = idy; j < n; j += offsety) {
        double rj = row[j];
        for (int i = idx; i < n; i += offsetx) {
            long long pos = (long long)j * n + i;
            if (i == k && j == k)  A[pos] = 1.0 / p;
            else if (i == k)       A[pos] = rj;
            else if (j == k)       A[pos] = -col[i] / p;
            else                   A[pos] -= col[i] * rj;
        }
    }
}

// ---- секундомер на cudaEvent ----------------------------------------------
struct Stopwatch { cudaEvent_t a, b; };

static void sw_init(Stopwatch *s) {
    CSC(cudaEventCreate(&s->a));
    CSC(cudaEventCreate(&s->b));
}
static void sw_start(Stopwatch *s) { CSC(cudaEventRecord(s->a, 0)); }
static double sw_stop(Stopwatch *s) {
    CSC(cudaEventRecord(s->b, 0));
    CSC(cudaEventSynchronize(s->b));
    float ms = 0.0f;
    CSC(cudaEventElapsedTime(&ms, s->a, s->b));
    return (double)ms;
}

static double now_ms() {
    using namespace std::chrono;
    return duration<double, std::milli>(steady_clock::now().time_since_epoch()).count();
}

// ---- быстрый ввод ----------------------------------------------------------
static unsigned char ibuf[1 << 16];
static size_t ipos = 0, ilen = 0;

static int next_char() {
    if (ipos == ilen) {
        ilen = fread(ibuf, 1, sizeof(ibuf), stdin);
        ipos = 0;
        if (ilen == 0) return -1;
    }
    return ibuf[ipos++];
}

static int read_token(char *tok, int cap) {
    int c;
    do { c = next_char(); } while (c == ' ' || c == '\n' || c == '\r' || c == '\t');
    if (c < 0) return 0;
    int len = 0;
    while (c >= 0 && c != ' ' && c != '\n' && c != '\r' && c != '\t') {
        if (len < cap - 1) tok[len++] = (char)c;
        c = next_char();
    }
    tok[len] = '\0';
    return 1;
}

static int read_double(double *x) {
    char tok[128];
    if (!read_token(tok, sizeof(tok))) return 0;
    char *end;
    *x = strtod(tok, &end);
    return *end == '\0';
}

static int read_int(long long *x) {
    char tok[128];
    if (!read_token(tok, sizeof(tok))) return 0;
    char *end;
    *x = strtoll(tok, &end, 10);
    return *end == '\0';
}

int main(int argc, char **argv) {
    int gx = 16, gy = 16, bx = 32, by = 32;
    bool noout = false, detail = false;
    for (int i = 1; i < argc; i++) {
        if (!strcmp(argv[i], "--grid") && i + 2 < argc) {
            gx = atoi(argv[i + 1]); gy = atoi(argv[i + 2]); i += 2;
        } else if (!strcmp(argv[i], "--block") && i + 2 < argc) {
            bx = atoi(argv[i + 1]); by = atoi(argv[i + 2]); i += 2;
        } else if (!strcmp(argv[i], "--noout")) {
            noout = true;
        } else if (!strcmp(argv[i], "--detail")) {
            detail = true;
        } else {
            fprintf(stderr, "ERROR: unknown or incomplete argument %s\n", argv[i]);
            return 0;
        }
    }
    if (gx < 1 || gy < 1 || bx < 1 || by < 1 || bx * by > 1024) {
        fprintf(stderr, "ERROR: invalid launch configuration\n");
        return 0;
    }
    dim3 grid2(gx, gy), block2(bx, by);

    CSC(cudaFree(0));   // инициализация контекста CUDA вне замеров

    double t_start = now_ms();

    long long nll;
    if (!read_int(&nll) || nll < 1 || nll > MAX_N) {
        fprintf(stderr, "ERROR: invalid matrix size (must be 1..%d)\n", MAX_N);
        return 0;
    }
    int n = (int)nll;
    size_t total = (size_t)n * (size_t)n;

    double *h = (double *)malloc(sizeof(double) * total);
    double *buf = (double *)malloc(sizeof(double) * (size_t)CHUNK * n);
    if (h == NULL || buf == NULL) {
        fprintf(stderr, "ERROR: not enough host memory\n");
        return 0;
    }
    for (int i0 = 0; i0 < n; i0 += CHUNK) {
        int rows = (n - i0 < CHUNK) ? (n - i0) : CHUNK;
        for (int r = 0; r < rows; r++)
            for (int j = 0; j < n; j++)
                if (!read_double(&buf[(size_t)r * n + j])) {
                    fprintf(stderr, "ERROR: cannot read matrix element (%d, %d)\n", i0 + r, j);
                    return 0;
                }
        for (int j = 0; j < n; j++)
            for (int r = 0; r < rows; r++)
                h[(size_t)j * n + i0 + r] = buf[(size_t)r * n + j];
    }
    double read_ms = now_ms() - t_start;

    double *dev_A = NULL, *dev_col = NULL, *dev_row = NULL;
    CSC(cudaMalloc(&dev_A, sizeof(double) * total));
    CSC(cudaMalloc(&dev_col, sizeof(double) * n));
    CSC(cudaMalloc(&dev_row, sizeof(double) * n));
    int *piv = (int *)malloc(sizeof(int) * n);

    Stopwatch sw_total, sw_h2d, sw_gauss, sw_d2h, sw_step;
    sw_init(&sw_total); sw_init(&sw_h2d); sw_init(&sw_gauss);
    sw_init(&sw_d2h); sw_init(&sw_step);
    double t_pivot = 0, t_swap = 0, t_prep = 0, t_update = 0;

    sw_start(&sw_total);

    sw_start(&sw_h2d);
    CSC(cudaMemcpy(dev_A, h, sizeof(double) * total, cudaMemcpyHostToDevice));
    double h2d_ms = sw_stop(&sw_h2d);

    sw_start(&sw_gauss);
    for (int k = 0; k < n; k++) {
        if (detail) sw_start(&sw_step);
        thrust::device_ptr<double> first = thrust::device_pointer_cast(dev_A + (size_t)k * n + k);
        thrust::device_ptr<double> last = thrust::device_pointer_cast(dev_A + (size_t)k * n + n);
        thrust::device_ptr<double> it = thrust::max_element(first, last, abs_less());
        int r = k + (int)(it - first);
        piv[k] = r;
        if (detail) t_pivot += sw_stop(&sw_step);

        if (r != k) {
            if (detail) sw_start(&sw_step);
            swap_rows<<<GRID1, BLOCK1>>>(dev_A, n, k, r);
            CSC(cudaGetLastError());
            if (detail) t_swap += sw_stop(&sw_step);
        }

        double p;
        CSC(cudaMemcpy(&p, dev_A + (size_t)k * n + k, sizeof(double), cudaMemcpyDeviceToHost));
        if (p == 0.0) {
            fprintf(stderr, "ERROR: matrix is singular\n");
            return 0;
        }

        if (detail) sw_start(&sw_step);
        prepare<<<GRID1, BLOCK1>>>(dev_A, dev_col, dev_row, n, k, p);
        CSC(cudaGetLastError());
        if (detail) t_prep += sw_stop(&sw_step);

        if (detail) sw_start(&sw_step);
        update<<<grid2, block2>>>(dev_A, dev_col, dev_row, n, k, p);
        CSC(cudaGetLastError());
        if (detail) t_update += sw_stop(&sw_step);
    }
    for (int k = n - 1; k >= 0; k--) {
        if (piv[k] != k) {
            swap_cols<<<GRID1, BLOCK1>>>(dev_A, n, k, piv[k]);
            CSC(cudaGetLastError());
        }
    }
    double gauss_ms = sw_stop(&sw_gauss);

    sw_start(&sw_d2h);
    CSC(cudaMemcpy(h, dev_A, sizeof(double) * total, cudaMemcpyDeviceToHost));
    double d2h_ms = sw_stop(&sw_d2h);

    double gpu_total_ms = sw_stop(&sw_total);

    CSC(cudaFree(dev_A));
    CSC(cudaFree(dev_col));
    CSC(cudaFree(dev_row));

    double write_ms = 0.0;
    if (!noout) {
        double t_w = now_ms();
        static char outbuf[1 << 20];
        setvbuf(stdout, outbuf, _IOFBF, sizeof(outbuf));
        for (int i0 = 0; i0 < n; i0 += CHUNK) {
            int rows = (n - i0 < CHUNK) ? (n - i0) : CHUNK;
            for (int j = 0; j < n; j++)
                for (int r = 0; r < rows; r++)
                    buf[(size_t)r * n + j] = h[(size_t)j * n + i0 + r];
            for (int r = 0; r < rows; r++)
                for (int j = 0; j < n; j++)
                    printf("%.10e%c", buf[(size_t)r * n + j], j + 1 < n ? ' ' : '\n');
        }
        fflush(stdout);
        write_ms = now_ms() - t_w;
    }

    fprintf(stderr,
            "TIMING= n=%d gx=%d gy=%d bx=%d by=%d h2d_ms=%.4f gauss_ms=%.4f d2h_ms=%.4f "
            "gpu_total_ms=%.4f pivot_ms=%.4f swap_ms=%.4f prep_ms=%.4f update_ms=%.4f "
            "read_ms=%.4f write_ms=%.4f\n",
            n, gx, gy, bx, by, h2d_ms, gauss_ms, d2h_ms, gpu_total_ms,
            t_pivot, t_swap, t_prep, t_update, read_ms, write_ms);

    free(h);
    free(buf);
    free(piv);
    return 0;
}


In [ ]:
%%writefile lab4_cpu.cpp
// ЛР №4, вариант 2. CPU-версия для сравнения с GPU (для отчёта, не для сдачи).
// Тот же алгоритм: Гаусс-Жордан "на месте" с выбором главного элемента по столбцу,
// один поток, матрица хранится по строкам (так цикл по j векторизуется компилятором).
//
// Сборка:  g++ -O2 -o lab4_cpu lab4_cpu.cpp
// Запуск:  ./lab4_cpu [--noout] < matrix.txt
// Результат замера -- одна строка CPUTIME=... в stderr.
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <chrono>

#define MAX_N 10000

static double now_ms() {
    using namespace std::chrono;
    return duration<double, std::milli>(steady_clock::now().time_since_epoch()).count();
}

static unsigned char ibuf[1 << 16];
static size_t ipos = 0, ilen = 0;

static int next_char() {
    if (ipos == ilen) {
        ilen = fread(ibuf, 1, sizeof(ibuf), stdin);
        ipos = 0;
        if (ilen == 0) return -1;
    }
    return ibuf[ipos++];
}

static int read_token(char *tok, int cap) {
    int c;
    do { c = next_char(); } while (c == ' ' || c == '\n' || c == '\r' || c == '\t');
    if (c < 0) return 0;
    int len = 0;
    while (c >= 0 && c != ' ' && c != '\n' && c != '\r' && c != '\t') {
        if (len < cap - 1) tok[len++] = (char)c;
        c = next_char();
    }
    tok[len] = '\0';
    return 1;
}

static int read_double(double *x) {
    char tok[128];
    if (!read_token(tok, sizeof(tok))) return 0;
    char *end;
    *x = strtod(tok, &end);
    return *end == '\0';
}

int main(int argc, char **argv) {
    bool noout = false;
    for (int i = 1; i < argc; i++) {
        if (!strcmp(argv[i], "--noout")) noout = true;
        else { fprintf(stderr, "ERROR: unknown argument %s\n", argv[i]); return 0; }
    }

    double t0 = now_ms();
    char tok[128];
    char *end;
    if (!read_token(tok, sizeof(tok))) { fprintf(stderr, "ERROR: invalid matrix size\n"); return 0; }
    long long nll = strtoll(tok, &end, 10);
    if (*end != '\0' || nll < 1 || nll > MAX_N) {
        fprintf(stderr, "ERROR: invalid matrix size (must be 1..%d)\n", MAX_N);
        return 0;
    }
    int n = (int)nll;
    double *A = (double *)malloc(sizeof(double) * (size_t)n * n);
    int *piv = (int *)malloc(sizeof(int) * n);
    if (A == NULL || piv == NULL) { fprintf(stderr, "ERROR: not enough host memory\n"); return 0; }
    for (size_t i = 0; i < (size_t)n * n; i++)
        if (!read_double(&A[i])) { fprintf(stderr, "ERROR: cannot read matrix element %zu\n", i); return 0; }
    double read_ms = now_ms() - t0;

    double t1 = now_ms();
    for (int k = 0; k < n; k++) {
        int r = k;
        double best = fabs(A[(size_t)k * n + k]);
        for (int i = k + 1; i < n; i++) {
            double v = fabs(A[(size_t)i * n + k]);
            if (v > best) { best = v; r = i; }
        }
        piv[k] = r;
        if (r != k)
            for (int j = 0; j < n; j++) {
                double t = A[(size_t)k * n + j];
                A[(size_t)k * n + j] = A[(size_t)r * n + j];
                A[(size_t)r * n + j] = t;
            }
        double p = A[(size_t)k * n + k];
        if (p == 0.0) { fprintf(stderr, "ERROR: matrix is singular\n"); return 0; }

        double *rk = A + (size_t)k * n;
        for (int j = 0; j < n; j++) rk[j] /= p;
        rk[k] = 1.0 / p;
        for (int i = 0; i < n; i++) {
            if (i == k) continue;
            double *ri = A + (size_t)i * n;
            double f = ri[k];
            for (int j = 0; j < n; j++) ri[j] -= f * rk[j];
            ri[k] = -f / p;
        }
    }
    for (int k = n - 1; k >= 0; k--) {
        if (piv[k] == k) continue;
        for (int i = 0; i < n; i++) {
            double t = A[(size_t)i * n + k];
            A[(size_t)i * n + k] = A[(size_t)i * n + piv[k]];
            A[(size_t)i * n + piv[k]] = t;
        }
    }
    double compute_ms = now_ms() - t1;

    double write_ms = 0.0;
    if (!noout) {
        double t2 = now_ms();
        static char outbuf[1 << 20];
        setvbuf(stdout, outbuf, _IOFBF, sizeof(outbuf));
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++)
                printf("%.10e%c", A[(size_t)i * n + j], j + 1 < n ? ' ' : '\n');
        fflush(stdout);
        write_ms = now_ms() - t2;
    }

    fprintf(stderr, "CPUTIME= n=%d compute_ms=%.4f read_ms=%.4f write_ms=%.4f\n",
            n, compute_ms, read_ms, write_ms);
    free(A);
    free(piv);
    return 0;
}


In [ ]:
%%writefile gen_matrix.py
#!/usr/bin/env python3
"""Генератор тестовых матриц для ЛР 4 (формат: n, затем n строк по n чисел).

Использование:
    python3 gen_matrix.py N out.txt [--seed S] [--kind KIND]

Виды матриц (--kind):
    random    элементы ~ N(0, 1); обратимая с вероятностью 1, число обусловленности ~ O(n)
    dominant  случайная матрица с диагональным преобладанием (хорошо обусловлена)
    pivot     строки dominant-матрицы случайно переставлены: на диагонали много нулей/малых
              элементов, выбор главного элемента обязателен
    hilbert   матрица Гильберта (очень плохо обусловлена; только для n <= 10)
    singular  вырожденная (две одинаковые строки). Внимание: в арифметике с плавающей точкой
              ведущий элемент обычно не получается ровно 0, поэтому программа может
              не выдать ERROR, а вернуть большие числа; ровно нулевой ведущий элемент
              гарантирован только для n = 1 (матрица [0]) и простых целочисленных случаев

Из Python:
    from gen_matrix import gen_matrix
    gen_matrix(1000, 'in_1000.txt', seed=1, kind='random')
"""
import argparse
import numpy as np


def make_matrix(n, seed=0, kind='random'):
    rng = np.random.default_rng(seed)
    if kind == 'random':
        return rng.standard_normal((n, n))
    if kind == 'dominant':
        A = rng.uniform(-1.0, 1.0, (n, n))
        A[np.diag_indices(n)] += np.sign(rng.standard_normal(n) + 1e-9) * (n + 1.0)
        return A
    if kind == 'pivot':
        A = make_matrix(n, seed, 'dominant')
        return A[rng.permutation(n)]
    if kind == 'hilbert':
        i = np.arange(n)
        return 1.0 / (i[:, None] + i[None, :] + 1.0)
    if kind == 'singular':
        A = rng.standard_normal((n, n))
        if n >= 2:
            A[-1] = A[0]
        else:
            A[0, 0] = 0.0
        return A
    raise ValueError(f'unknown kind: {kind}')


def gen_matrix(n, path, seed=0, kind='random', block_rows=256):
    """Записывает матрицу в файл блоками (экономит память при больших n)."""
    A = make_matrix(n, seed, kind)
    with open(path, 'w') as f:
        f.write(f'{n}\n')
        for i0 in range(0, n, block_rows):
            np.savetxt(f, A[i0:i0 + block_rows], fmt='%.15e')
    return A


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('n', type=int)
    ap.add_argument('out')
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--kind', default='random',
                    choices=['random', 'dominant', 'pivot', 'hilbert', 'singular'])
    a = ap.parse_args()
    gen_matrix(a.n, a.out, a.seed, a.kind)
    print(f'{a.kind} matrix {a.n}x{a.n} written to {a.out}')


## 2. Компиляция

In [ ]:
%%shell
set -e
nvcc -arch=sm_75 -Werror cross-execution-space-call -lm -o lab4 lab4.cu
nvcc -arch=sm_75 -O2 -o lab4_prof lab4_prof.cu
g++ -O2 -o lab4_cpu lab4_cpu.cpp
ls -l lab4 lab4_prof lab4_cpu

## 3. Параметры и генерация тестов

Матрицы генерируются в формате задания (`n`, затем `n` строк по `n` чисел). Вид матрицы `KIND`:
`random` (нормальные элементы), `dominant` (диагональное преобладание), `pivot` (строки перемешаны, выбор главного элемента обязателен).
Генерация больших матриц занимает до минуты, файлы повторно не создаются.

In [ ]:
import subprocess, re, os, time, statistics
import numpy as np
import pandas as pd
from gen_matrix import gen_matrix

# ---------------- Параметры ----------------
SIZES     = [100, 250, 500, 1000, 2000, 3000]   # размеры матриц для таблицы
CPU_MAX_N = 2000      # CPU-версию запускаем только до этого n (она медленная)
REPS      = 3         # число повторов для GPU, берётся медиана
N_STUDY   = 1000      # размер матрицы для исследования конфигурации сетки/блока
KIND      = 'random'  # random | dominant | pivot
# --------------------------------------------

def matrix_path(n):
    return f'in_{KIND}_{n}.txt'

for n in sorted(set(SIZES + [N_STUDY])):
    if not os.path.exists(matrix_path(n)):
        t = time.time()
        gen_matrix(n, matrix_path(n), seed=n, kind=KIND)
        print(f'n={n}: матрица сгенерирована за {time.time() - t:.1f} с')

KV = re.compile(r'(\w+)=(-?\d+(?:\.\d+)?(?:e[+-]?\d+)?)\b')

def run(cmd, path):
    with open(path) as f:
        r = subprocess.run(cmd, stdin=f, capture_output=True, text=True)
    if 'ERROR' in r.stderr:
        raise RuntimeError(f"{' '.join(cmd)}: {r.stderr.strip()}")
    return r

def metrics(r, tag):
    line = [l for l in r.stderr.splitlines() if l.startswith(tag)][-1]
    return {k: float(v) for k, v in KV.findall(line)}

def median(values):
    return statistics.median(values)

def md_table(df):
    cols = list(df.columns)
    out = ['| ' + ' | '.join(cols) + ' |', '|' + '|'.join(['---'] * len(cols)) + '|']
    for row in df.itertuples(index=False):
        out.append('| ' + ' | '.join('' if pd.isna(v) else (f'{v:.2f}' if isinstance(v, (float, np.floating)) else str(v)) for v in row) + ' |')
    return '\n'.join(out)

## 4. Проверка корректности

GPU-версия (`lab4`), профилирующая (`lab4_prof`) и CPU-версия (`lab4_cpu`) сравниваются с `numpy.linalg.inv`.
Погрешность около 1e-11 объясняется тем, что вывод округляется до 10 значащих цифр.

In [ ]:
import tempfile

def parse_matrix(text):
    return np.array([[float(x) for x in line.split()] for line in text.strip().split('\n')])

rows = []
for kind in ('random', 'pivot'):
    for n in (1, 2, 7, 64, 130, 300):
        path = f'check_{kind}_{n}.txt'
        A = gen_matrix(n, path, seed=100 + n, kind=kind)
        ref = np.linalg.inv(A)
        row = {'вид': kind, 'n': n}
        for name, cmd in (('lab4', ['./lab4']), ('lab4_prof', ['./lab4_prof']), ('lab4_cpu', ['./lab4_cpu'])):
            r = run(cmd, path)
            rel = np.abs(parse_matrix(r.stdout) - ref).max() / np.abs(ref).max()
            row[name] = f'{rel:.1e}'
            row[name + ' ok'] = rel < max(1e-9, np.linalg.cond(A) * 1e-14)
        rows.append(row)
check = pd.DataFrame(rows)
print(check.to_string(index=False))
print('\nВсе проверки пройдены:', bool(check[[c for c in check.columns if c.endswith(' ok')]].all().all()))

## 5. Таблица «CPU и GPU»

* **GPU Gauss** время работы алгоритма на видеокарте, измеренное событиями `cudaEvent` (без копирования данных).
* **GPU total** то же вместе с копированием матрицы на видеокарту и обратно.
* **CPU** время работы той же схемы Гаусса–Жордана на одном ядре процессора (`g++ -O2`).
* Ввод и вывод текстом в замеры не входят: они одинаково медленные для обеих версий.

In [ ]:
rows = []
for n in SIZES:
    p = matrix_path(n)
    run(['./lab4_prof', '--noout'], p)   # прогрев видеокарты
    g = [metrics(run(['./lab4_prof', '--noout'], p), 'TIMING=') for _ in range(REPS)]
    row = {
        'n': n,
        'GPU Gauss, мс': median([x['gauss_ms'] for x in g]),
        'GPU total, мс': median([x['gpu_total_ms'] for x in g]),
        'GPU H2D, мс':   median([x['h2d_ms'] for x in g]),
        'GPU D2H, мс':   median([x['d2h_ms'] for x in g]),
    }
    if n <= CPU_MAX_N:
        reps_cpu = REPS if n <= 1000 else 1
        c = [metrics(run(['./lab4_cpu', '--noout'], p), 'CPUTIME=')['compute_ms'] for _ in range(reps_cpu)]
        row['CPU, мс'] = median(c)
        row['Ускорение CPU/GPU Gauss'] = row['CPU, мс'] / row['GPU Gauss, мс']
        row['Ускорение CPU/GPU total'] = row['CPU, мс'] / row['GPU total, мс']
    rows.append(row)
    print(f'n={n} готово')
table = pd.DataFrame(rows)
cols = ['n', 'CPU, мс', 'GPU Gauss, мс', 'GPU total, мс', 'Ускорение CPU/GPU Gauss', 'Ускорение CPU/GPU total', 'GPU H2D, мс', 'GPU D2H, мс']
table = table[[c for c in cols if c in table.columns]]
table.to_csv('table_cpu_gpu.csv', index=False)
print(table.round(2).to_string(index=False))
print('\n--- Markdown для отчёта ---')
print(md_table(table))

## 6. На что тратится время (по фазам)

Режим `--detail` ставит события вокруг каждой фазы на каждом шаге метода, поэтому общее время чуть больше, чем в таблице выше.
Доли показывают, какая часть работы доминирует (для больших n ожидается ядро `update`).

In [ ]:
rows = []
for n in SIZES:
    d = metrics(run(['./lab4_prof', '--noout', '--detail'], matrix_path(n)), 'TIMING=')
    phases = {'поиск максимума (Thrust)': d['pivot_ms'], 'обмен строк': d['swap_ms'],
              'подготовка (prepare)': d['prep_ms'], 'обновление (update)': d['update_ms']}
    s = sum(phases.values())
    row = {'n': n, 'сумма фаз, мс': s}
    row.update({k + ', %': 100.0 * v / s for k, v in phases.items()})
    rows.append(row)
phases_df = pd.DataFrame(rows)
phases_df.to_csv('table_phases.csv', index=False)
print(phases_df.round(1).to_string(index=False))
print('\n--- Markdown для отчёта ---')
print(md_table(phases_df))

## 7. Конфигурация сетки и блока ядра `update`

Для матрицы `N_STUDY × N_STUDY` перебираются разные `<<<grid, block>>>`. Число потоков фиксировано внутри каждого запуска и не зависит от n,
поэтому сетка покрывает матрицу циклом с шагом. Так видно, как размер сетки и блока влияет на время.

In [ ]:
CONFIGS = [  # (grid_x, grid_y, block_x, block_y)
    (1, 1, 32, 1), (1, 1, 32, 8), (1, 1, 32, 32),
    (4, 4, 16, 16), (8, 8, 32, 8), (16, 16, 16, 16),
    (16, 16, 32, 32), (32, 32, 32, 16), (64, 64, 16, 16),
]
rows = []
for gx, gy, bx, by in CONFIGS:
    cmd = ['./lab4_prof', '--noout', '--detail', '--grid', str(gx), str(gy), '--block', str(bx), str(by)]
    d = metrics(run(cmd, matrix_path(N_STUDY)), 'TIMING=')
    rows.append({'grid': f'{gx}x{gy}', 'block': f'{bx}x{by}', 'всего потоков': gx * gy * bx * by,
                 'update, мс': d['update_ms'], 'Gauss, мс': d['gauss_ms']})
conf = pd.DataFrame(rows).sort_values('update, мс').reset_index(drop=True)
conf.to_csv('table_configs.csv', index=False)
print(f'n = {N_STUDY}')
print(conf.round(2).to_string(index=False))
print('\n--- Markdown для отчёта ---')
print(md_table(conf))

## 8. График

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
t = table
if 'CPU, мс' in t:
    ax[0].plot(t['n'], t['CPU, мс'], 'o-', label='CPU (1 ядро)')
ax[0].plot(t['n'], t['GPU Gauss, мс'], 's-', label='GPU (алгоритм)')
ax[0].plot(t['n'], t['GPU total, мс'], '^-', label='GPU (с копированием)')
ax[0].set_xscale('log'); ax[0].set_yscale('log')
ax[0].set_xlabel('n'); ax[0].set_ylabel('время, мс'); ax[0].set_title('Обращение матрицы n x n')
ax[0].grid(True, which='both', alpha=0.3); ax[0].legend()
if 'Ускорение CPU/GPU Gauss' in t:
    s = t.dropna(subset=['Ускорение CPU/GPU Gauss'])
    ax[1].plot(s['n'], s['Ускорение CPU/GPU Gauss'], 's-', label='без копирования')
    ax[1].plot(s['n'], s['Ускорение CPU/GPU total'], '^-', label='с копированием')
    ax[1].axhline(1.0, color='gray', ls='--')
    ax[1].set_xscale('log'); ax[1].set_xlabel('n'); ax[1].set_ylabel('ускорение, раз')
    ax[1].set_title('Ускорение GPU относительно CPU'); ax[1].grid(True, which='both', alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.savefig('time_vs_n.png', dpi=150); plt.show()

## 9. Профайлер (замена `nvprof`)

`nvprof` в современных CUDA не поддерживается, вместо него используются Nsight Systems (`nsys`) и Nsight Compute (`ncu`).
Если ячейка пишет, что `nsys` не найден, то в этой среде он не установлен: тогда в отчёте можно сослаться на замеры через `cudaEvent` из разделов 5 и 6.

In [ ]:
!which nsys ncu nvprof || echo "nsys / ncu / nvprof не найдены в PATH"
!nsys profile --stats=true --force-overwrite true -o prof_report ./lab4_prof --noout < {matrix_path(N_STUDY)} 2>&1 | head -60

## 10. Что вставить в отчёт

* Таблица из раздела 5 (CPU, GPU, ускорение) и график `time_vs_n.png` (файл слева в панели).
* Таблица по фазам из раздела 6: какая часть времени уходит на поиск максимума, обмен строк и обновление.
* Таблица из раздела 7: влияние конфигурации сетки и блока.
* Вывод `nsys` из раздела 9 (если доступен).
* Описание оборудования из первой ячейки (видеокарта и процессор Colab).

Файлы `table_*.csv` и `time_vs_n.png` можно скачать из панели слева.